In [1]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Задание

1) Реализовать методы `greedy_sampling` и `generate` (1 балл)
2) Реализовать метод `random_sampling` и поддержать его в `generate` (1 балл)
3) Реализовать метод `_beam_search_generate` и поддержать его в `generate` (2 балла)
4) Реализовать методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержать их в `generate` (1 балл)  
Все методы необходимо реализовать через векторные операции в torch/numpy везде где это возможно

In [5]:
class Model:
    def __init__(self, model_name: str = "gpt2"):
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size

    def greedy_sampling(self, logits: torch.Tensor) -> int:
        next_token_id = torch.argmax(logits, dim=-1).item()
        return next_token_id    

    def random_sampling(self, logits: torch.Tensor) -> int:
        probs = F.softmax(logits, dim=-1)
        next_token_id = torch.multinomial(probs, num_samples=1).item()
        return next_token_id

    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int,
        num_beams: int
    ) -> str:
        device = next(self.model.parameters()).device
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
        beams = [(input_ids, 0.0)]

        for _ in range(max_length):
            candidates = []
            for seq, score in beams:
                with torch.no_grad():
                    outputs = self.model(seq)
                    logits = outputs.logits[:, -1, :]
                    log_probs = F.log_softmax(logits, dim=-1)

                top_log_probs, top_indices = torch.topk(log_probs, num_beams, dim=-1)
                for log_p, idx in zip(top_log_probs[0], top_indices[0]):
                    new_seq = torch.cat([seq, idx.unsqueeze(0).unsqueeze(0)], dim=1)
                    new_score = score + log_p.item()
                    candidates.append((new_seq, new_score))

            beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:num_beams]
            if all(self.tokenizer.eos_token_id in seq for seq, _ in beams):
                break

        best_seq = max(beams, key=lambda x: x[1])[0]
        return self.tokenizer.decode(best_seq[0], skip_special_tokens=True)

    def apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        if temperature <= 0:
            raise ValueError("Температура должна быть > 0")
        adjusted_logits = logits / temperature
        return adjusted_logits


    def apply_top_k(self, logits: torch.Tensor, top_k: int = 50) -> torch.Tensor:
        if top_k <= 0:
            return logits
        topk_values, _ = torch.topk(logits, top_k)
        min_topk = topk_values[..., -1, None]
        filtered_logits = torch.where(logits < min_topk, torch.tensor(float('-inf')).to(logits.device), logits)
        return filtered_logits

    def apply_top_p(self, logits: torch.Tensor, top_p: float = 0.9) -> torch.Tensor:
        if top_p >= 1.0 or top_p <= 0.0:
            return logits

        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        probs = F.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(probs, dim=-1)
        cutoff = cumulative_probs > top_p
        cutoff[..., 1:] = cutoff[..., :-1].clone()
        cutoff[..., 0] = False

        sorted_logits[cutoff] = float('-inf')
        unsorted_logits = torch.zeros_like(logits).scatter_(dim=-1, index=sorted_indices, src=sorted_logits)
        return unsorted_logits

    def generate(
        self,
        prompt: str,
        max_length: int = 50,
        strategy: str = "greedy",    # greedy | random | beam
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        num_beams: int = 3
    ) -> str:
        device = next(self.model.parameters()).device
        self.model.eval()
        if strategy == "beam":
            return self._beam_search_generate(
                prompt=prompt,
                max_length=max_length,
                num_beams=num_beams
            )

        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
        eos_id = self.tokenizer.eos_token_id

        with torch.no_grad():
            for _ in range(max_length):
                outputs = self.model(input_ids)
                logits_last = outputs.logits[:, -1, :]
                logits_vec = logits_last.squeeze(0)

                if strategy == "greedy":
                    next_token_int = self.greedy_sampling(logits_vec)
                elif strategy == "random":
                    if temperature != 1.0:
                        logits_vec = self.apply_temperature(logits_vec, temperature=temperature)
                    if top_k > 0:
                        logits_vec = self.apply_top_k(logits_vec, top_k=top_k)
                    if 0.0 < top_p < 1.0:
                        logits_vec = self.apply_top_p(logits_vec, top_p=top_p)

                    next_token_int = self.random_sampling(logits_vec)
                else:
                    raise ValueError("strategy должен быть 'greedy', 'random' или 'beam'")

                next_token_id = torch.tensor([[next_token_int]], device=device, dtype=input_ids.dtype)
                input_ids = torch.cat([input_ids, next_token_id], dim=1)

                if eos_id is not None and next_token_int == int(eos_id):
                    break

        text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
        return text

In [11]:
prompt = "Aldiyar loves burgers"


model = Model()
print(f'Промпт: "{prompt}"')

try:
    out = model.generate(prompt, max_length=25, strategy="greedy")
    print("\n[greedy]\n", out)

    out = model.generate(prompt, max_length=25, strategy="random", temperature=1.0)
    print("\n[random, temp=1.0]\n", out)

    out = model.generate(prompt, max_length=25, strategy="random", temperature=0.7, top_k=50)
    print("\n[random, temp=0.7, top_k=50]\n", out)

    out = model.generate(prompt, max_length=25, strategy="random", top_p=0.9)
    print("\n[random, top_p=0.9]\n", out)

    out = model.generate(prompt, max_length=25, strategy="beam", num_beams=3)
    print("\n[beam, num_beams=3]\n", out)

except Exception as e:
    print("\nОшибка при генерации:", e)
    print("Убедитесь, что установлен пакет transformers и модель доступна (через интернет или из кеша).")


Промпт: "Aldiyar loves burgers"

[greedy]
 Aldiyar loves burgers, but he's not a fan of the idea of a burger that's too big.

"I think it's

[random, temp=1.0]
 Aldiyar loves burgers. He saw him bring them to my hotel seven years ago once and they were delicious—the best burger I'd ever had

[random, temp=0.7, top_k=50]
 Aldiyar loves burgers, and he's been very involved in local businesses. That's why he's been a big supporter of the city's food

[random, top_p=0.9]
 Aldiyar loves burgers and pasta, and he also likes to eat sports foods that are jammy and sour, like pasta. "That kind of

[beam, num_beams=3]
 Aldiyar loves burgers.

"I'm a big fan of burgers," he said. "I've always been a big fan of burgers
